In [2]:
import pandas as pd

In [3]:
full_annot = pd.read_csv('./hg38_genes_and_exons.txt', sep='\t')
full_annot=full_annot.rename(columns={'#NAME': 'NAME',}, inplace=False)
full_annot

,NAME,CHROM,STRAND,TX_START,TX_END,EXON_START,EXON_END
0,OR4F5,1,+,69090,70008,"69090,","70008,"
1,OR4F16,1,-,685715,686654,"685715,","686654,"
2,SAMD11,1,+,925737,944575,"925737,925921,930154,931038,935771,939039,9392...","925800,926013,930336,931089,935896,939129,9394..."
3,NOC2L,1,-,944203,959290,"944203,945056,945517,946172,946401,948130,9484...","944800,945146,945653,946286,946545,948232,9486..."
4,KLHL17,1,+,960586,965715,"960586,961292,961628,961825,962354,962703,9631...","960800,961552,961750,962047,962471,962917,9632..."
...,...,...,...,...,...,...,...
19300,BPY2B,Y,+,24618003,24639207,"24618003,24618309,24621497,24626043,24628221,2...","24618034,24618432,24621634,24626162,24628339,2..."
19301,DAZ3,Y,-,24763068,24813479,"24763068,24768898,24773569,24774349,24784136,2...","24764925,24768997,24773604,24774421,24784208,2..."
19302,DAZ4,Y,+,24834126,24905240,"24834126,24840730,24841151,24841803,24842291,2...","24834129,24840877,24841243,24841855,24842355,2..."
19303,BPY2C,Y,-,25030900,25052104,"25030900,25031316,25037991,25038808,25041768,2...","25031222,25031441,25038116,25038914,25041886,2..."


In [4]:
# Make the gene bed file not needed for new version of dataset
gene_bed = full_annot[['CHROM','TX_START','TX_END','NAME','STRAND',]].rename(columns={
    'CHROM': '#CHROM',
    'TX_START': 'START', 
    'TX_END': 'END'
    }, inplace=False)
pad = 10000
gene_bed['START'] = gene_bed['START'].astype(int)-pad
gene_bed['END'] = gene_bed['END'].astype(int)+pad
gene_bed['#CHROM'] = 'chr' + gene_bed['#CHROM'].astype(str)
gene_bed['SCORE'] = 1
gene_bed=gene_bed[['#CHROM','START','END','NAME','SCORE','STRAND']]
gene_bed.to_csv('gene_bed.bed', sep='\t', index=False,header=False)
gene_bed[gene_bed['#CHROM']=='chr22'].to_csv('gene_bed_chr22.bed', sep='\t', index=False,header=False)

display(gene_bed)




,#CHROM,START,END,NAME,SCORE,STRAND
0,chr1,59090,80008,OR4F5,1,+
1,chr1,675715,696654,OR4F16,1,-
2,chr1,915737,954575,SAMD11,1,+
3,chr1,934203,969290,NOC2L,1,-
4,chr1,950586,975715,KLHL17,1,+
...,...,...,...,...,...,...
19300,chrY,24608003,24649207,BPY2B,1,+
19301,chrY,24753068,24823479,DAZ3,1,-
19302,chrY,24824126,24915240,DAZ4,1,+
19303,chrY,25020900,25062104,BPY2C,1,-


In [4]:
#make the exon bed file Needed for making annotation tracks
exon_bed = full_annot[['CHROM','EXON_START','EXON_END','STRAND','NAME',]].rename(columns={
    'CHROM': '#CHROM',
    # 'EXON_START': 'START', 
    # 'EXON_END': 'END'
    }, inplace=False)
exon_bed['#CHROM'] = 'chr' + exon_bed['#CHROM'].astype(str)
# exon_bed.to_csv('exon_bed.bed', sep='\t', index=False)

def get_p5_sites(row):
    if row['STRAND'] == '+':
        sites = row['EXON_START'].split(',')[1:]
        
    else:
        sites = row['EXON_END'].split(',')[:-1]
    
    sites = [int(site) if site != '' else None for site in sites]
    return(sites)

def get_p3_sites(row):
    if row['STRAND'] == '+':
        sites = row['EXON_END'].split(',')[:-1]
    else:
        sites = row['EXON_START'].split(',')[1:]
    sites = [int(site) if site != '' else None for site in sites]
    return(sites)

exon_bed["5'ss START"] = exon_bed.apply(get_p5_sites, axis=1)
exon_bed["3'ss START"] = exon_bed.apply(get_p3_sites, axis=1)

display(exon_bed)

,#CHROM,EXON_START,EXON_END,STRAND,NAME,5'ss START,3'ss START
0,chr1,"69090,","70008,",+,OR4F5,[None],[70008]
1,chr1,"685715,","686654,",-,OR4F16,[686654],[None]
2,chr1,"925737,925921,930154,931038,935771,939039,9392...","925800,926013,930336,931089,935896,939129,9394...",+,SAMD11,"[925921, 930154, 931038, 935771, 939039, 93927...","[925800, 926013, 930336, 931089, 935896, 93912..."
3,chr1,"944203,945056,945517,946172,946401,948130,9484...","944800,945146,945653,946286,946545,948232,9486...",-,NOC2L,"[944800, 945146, 945653, 946286, 946545, 94823...","[945056, 945517, 946172, 946401, 948130, 94848..."
4,chr1,"960586,961292,961628,961825,962354,962703,9631...","960800,961552,961750,962047,962471,962917,9632...",+,KLHL17,"[961292, 961628, 961825, 962354, 962703, 96310...","[960800, 961552, 961750, 962047, 962471, 96291..."
...,...,...,...,...,...,...,...
19300,chrY,"24618003,24618309,24621497,24626043,24628221,2...","24618034,24618432,24621634,24626162,24628339,2...",+,BPY2B,"[24618309, 24621497, 24626043, 24628221, 24631...","[24618034, 24618432, 24621634, 24626162, 24628..."
19301,chrY,"24763068,24768898,24773569,24774349,24784136,2...","24764925,24768997,24773604,24774421,24784208,2...",-,DAZ3,"[24764925, 24768997, 24773604, 24774421, 24784...","[24768898, 24773569, 24774349, 24784136, 24786..."
19302,chrY,"24834126,24840730,24841151,24841803,24842291,2...","24834129,24840877,24841243,24841855,24842355,2...",+,DAZ4,"[24840730, 24841151, 24841803, 24842291, 24842...","[24834129, 24840877, 24841243, 24841855, 24842..."
19303,chrY,"25030900,25031316,25037991,25038808,25041768,2...","25031222,25031441,25038116,25038914,25041886,2...",-,BPY2C,"[25031222, 25031441, 25038116, 25038914, 25041...","[25031316, 25037991, 25038808, 25041768, 25043..."


In [9]:
#make the p5 bed file needed for making annotation tracks
p5_bed = exon_bed[['#CHROM', "5'ss START", 'STRAND','NAME']]
p5_bed = p5_bed.explode(column=["5'ss START"])
p5_bed["5'ss END"] = p5_bed["5'ss START"] + 1
p5_bed = p5_bed[['#CHROM', "5'ss START", "5'ss END", 'STRAND','NAME']].rename(columns={
    "5'ss START": 'START',
    "5'ss END": 'END',
    }, inplace=False).dropna(inplace=False)
p5_bed['SCORE'] = 1
p5_bed=p5_bed[['#CHROM','START','END','NAME','SCORE','STRAND']]
p5_bed.to_csv('p5_bed.bed', sep='\t', index=False)
p5_bed[p5_bed['#CHROM']=='chr22'].to_csv('p5_chr22_bed.bed', sep='\t', index=False, header=False)

p5_bed

,#CHROM,START,END,NAME,SCORE,STRAND
1,chr1,686654,686655,OR4F16,1,-
2,chr1,925921,925922,SAMD11,1,+
2,chr1,930154,930155,SAMD11,1,+
2,chr1,931038,931039,SAMD11,1,+
2,chr1,935771,935772,SAMD11,1,+
...,...,...,...,...,...,...
19303,chrY,25041886,25041887,BPY2C,1,-
19303,chrY,25044064,25044065,BPY2C,1,-
19303,chrY,25048610,25048611,BPY2C,1,-
19303,chrY,25051798,25051799,BPY2C,1,-


In [10]:
#make the p3 bed file needed for making annotation tracks
p3_bed = exon_bed[['#CHROM', "3'ss START", 'STRAND','NAME']]
p3_bed = p3_bed.explode(column=["3'ss START"])
p3_bed["3'ss END"] = p3_bed["3'ss START"] + 1
p3_bed = p3_bed[['#CHROM', "3'ss START", "3'ss END", 'STRAND','NAME']].rename(columns={
    "3'ss START": 'START',
    "3'ss END": 'END',
    }, inplace=False).dropna(inplace=False)

p3_bed['SCORE'] = 1
p3_bed=p3_bed[['#CHROM','START','END','NAME','SCORE','STRAND']]
p3_bed.to_csv('p3_bed.bed', sep='\t', index=False)
p3_bed[p3_bed['#CHROM']=='chr22'].to_csv('p3_chr22_bed.bed', sep='\t', index=False, header=False)

p3_bed

,#CHROM,START,END,NAME,SCORE,STRAND
0,chr1,70008,70009,OR4F5,1,+
2,chr1,925800,925801,SAMD11,1,+
2,chr1,926013,926014,SAMD11,1,+
2,chr1,930336,930337,SAMD11,1,+
2,chr1,931089,931090,SAMD11,1,+
...,...,...,...,...,...,...
19303,chrY,25043945,25043946,BPY2C,1,-
19303,chrY,25048473,25048474,BPY2C,1,-
19303,chrY,25051675,25051676,BPY2C,1,-
19303,chrY,25052073,25052074,BPY2C,1,-
